# Semana 4: Sistemas de Ecuaciones y la Base de la Regresión
## Notebook de Ejercicios (NB2) - Dataset California Housing

### Propósito de la sesión
Aplicar los conceptos de sistemas sobredeterminados y mínimos cuadrados a un problema real de regresión: predecir el valor medio de las viviendas en distritos de California. Implementaremos la **ecuación normal** desde cero, compararemos con la implementación de scikit-learn y evaluaremos el modelo usando métricas estándar.

### Objetivos de aprendizaje
*   Cargar y explorar el dataset **California Housing**.
*   Implementar la **ecuación normal** $\hat{\beta} = (X^T X)^{-1} X^T y$ usando solo NumPy.
*   Comparar nuestros coeficientes con los de `sklearn.linear_model.LinearRegression`.
*   Evaluar el modelo usando **MSE** (Error Cuadrático Medio) y **$R^2$** (coeficiente de determinación).
*   Visualizar las predicciones vs valores reales y analizar los residuos.

## Configuración Inicial

Importamos las librerías necesarias y cargamos el dataset California Housing.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 5)

# Cargamos el dataset
print("Cargando dataset California Housing...")
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = housing.feature_names

print(f"\n✅ Dataset cargado correctamente.")
print(f"Shape de X (matriz de características): {X.shape}")
print(f"Shape de y (vector objetivo): {y.shape}")
print(f"\nCaracterísticas: {feature_names}")
print(f"Descripción del target: {housing.DESCR.splitlines()[0]}")

### Exploración inicial de los datos

Visualicemos las primeras filas y algunas estadísticas descriptivas.

In [ ]:
# Creamos un DataFrame para facilitar la exploración
df = pd.DataFrame(X, columns=feature_names)
df['MedHouseVal'] = y  # Valor medio de la vivienda (target)

print("Primeras 5 filas del dataset:")
display(df.head())

print("\nEstadísticas descriptivas:")
display(df.describe())

# Verificamos si hay valores nulos
print(f"\nValores nulos por columna:\n{df.isnull().sum()}")

### División en entrenamiento y prueba

Separamos los datos en conjuntos de entrenamiento (80%) y prueba (20%) para evaluar el modelo de manera justa.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape y_test: {y_test.shape}")

---
## Actividad 1: Implementar la ecuación normal desde cero

Recordemos que la **ecuación normal** para regresión lineal (con intercepto) es:

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

Donde $X$ debe incluir una columna de **unos** para el término de intercepto ($\beta_0$).

In [ ]:
# Añadimos una columna de unos a X_train para el intercepto
X_train_with_ones = np.column_stack([np.ones(X_train.shape[0]), X_train])

print(f"Shape de X_train con unos: {X_train_with_ones.shape}")
print(f"Primeras 5 filas de X_train_with_ones (primer columna es 1):\n{X_train_with_ones[:5, :5]}")

# Implementamos la ecuación normal
def ecuacion_normal(X, y):
    """
    Calcula los coeficientes de regresión lineal usando la ecuación normal.
    X: matriz de características (con columna de unos para intercepto)
    y: vector objetivo
    """
    XT = X.T
    XTX_inv = np.linalg.inv(XT @ X)
    beta = XTX_inv @ (XT @ y)
    return beta

# Calculamos los coeficientes
beta_normal = ecuacion_normal(X_train_with_ones, y_train)

print("\n🔷 Coeficientes calculados con ecuación normal:")
print(f"Intercepto (β₀): {beta_normal[0]:.6f}")
for i, name in enumerate(feature_names):
    print(f"{name} (β_{i+1}): {beta_normal[i+1]:.6f}")

### Verificación de la solución

Verifiquemos que $X^T X \hat{\beta} \approx X^T y$ (la ecuación normal debe satisfacerse).

In [ ]:
XTX = X_train_with_ones.T @ X_train_with_ones
XTy = X_train_with_ones.T @ y_train
lado_izquierdo = XTX @ beta_normal
lado_derecho = XTy

diferencia = np.max(np.abs(lado_izquierdo - lado_derecho))
print(f"Máxima diferencia entre XTX·β y XTy: {diferencia:.2e}")

if diferencia < 1e-10:
    print("✅ La ecuación normal se satisface correctamente.")
else:
    print("⚠️ Hay una discrepancia numérica (puede deberse a redondeo).")

---
## Actividad 2: Comparar con `sklearn.linear_model.LinearRegression`

Ahora usamos la implementación optimizada de scikit-learn y comparamos los coeficientes.

In [ ]:
# Creamos y entrenamos el modelo de sklearn
model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

# Obtenemos coeficientes
intercept_sk = model_sklearn.intercept_
coef_sk = model_sklearn.coef_

print("🔶 Coeficientes de sklearn:")
print(f"Intercepto: {intercept_sk:.6f}")
for i, name in enumerate(feature_names):
    print(f"{name}: {coef_sk[i]:.6f}")

# Comparamos con nuestra implementación
print("\n📊 Comparación de coeficientes:")
print(f"{'Característica':<20} {'Normal':<15} {'sklearn':<15} {'Diferencia':<15}")
print("-" * 65)
print(f"{'Intercepto':<20} {beta_normal[0]:<15.6f} {intercept_sk:<15.6f} {beta_normal[0] - intercept_sk:<15.2e}")
for i, name in enumerate(feature_names):
    diff = beta_normal[i+1] - coef_sk[i]
    print(f"{name:<20} {beta_normal[i+1]:<15.6f} {coef_sk[i]:<15.6f} {diff:<15.2e}")

# Verificamos si son prácticamente iguales
coef_normal_completo = np.concatenate([[beta_normal[0]], beta_normal[1:]])
coef_sk_completo = np.concatenate([[intercept_sk], coef_sk])
max_diff = np.max(np.abs(coef_normal_completo - coef_sk_completo))
print(f"\n📌 Máxima diferencia absoluta entre coeficientes: {max_diff:.2e}")

if max_diff < 1e-10:
    print("✅ Los coeficientes son prácticamente idénticos.")
else:
    print("⚠️ Hay diferencias significativas. Revisa tu implementación.")

---
## Actividad 3: Evaluar el modelo con MSE y $R^2$

Ahora evaluaremos ambos modelos (el nuestro y el de sklearn) en el conjunto de prueba usando dos métricas:

*   **MSE (Mean Squared Error)**: $\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$
*   **$R^2$ (Coeficiente de determinación)**: $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$

In [ ]:
# Función para calcular métricas
def evaluar_modelo(beta, X_train, X_test, y_train, y_test, feature_names, nombre="Modelo"):
    """
    Evalúa un modelo de regresión lineal.
    beta debe incluir el intercepto como primer elemento.
    """
    # Añadimos columna de unos a X_test para predicciones
    X_test_with_ones = np.column_stack([np.ones(X_test.shape[0]), X_test])
    
    # Predicciones
    y_pred_train = np.column_stack([np.ones(X_train.shape[0]), X_train]) @ beta
    y_pred_test = X_test_with_ones @ beta
    
    # Métricas
    mse_train = mean_squared_error(y_train, y_pred_train)
    mse_test = mean_squared_error(y_test, y_pred_test)
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    
    print(f"\n📊 {nombre}:")
    print(f"  MSE Train: {mse_train:.6f}")
    print(f"  MSE Test:  {mse_test:.6f}")
    print(f"  R² Train:  {r2_train:.6f}")
    print(f"  R² Test:   {r2_test:.6f}")
    
    return y_pred_test, mse_test, r2_test

# Evaluamos nuestro modelo (ecuación normal)
y_pred_normal, mse_normal, r2_normal = evaluar_modelo(
    beta_normal, X_train, X_test, y_train, y_test, feature_names, "Modelo Ecuación Normal"
)

# Evaluamos modelo de sklearn
y_pred_sk = model_sklearn.predict(X_test)
mse_sk = mean_squared_error(y_test, y_pred_sk)
r2_sk = r2_score(y_test, y_pred_sk)
print(f"\n📊 Modelo sklearn:")
print(f"  MSE Test:  {mse_sk:.6f}")
print(f"  R² Test:   {r2_sk:.6f}")

### Visualización de predicciones vs valores reales

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_normal, alpha=0.5, edgecolors='k', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Valores Reales')
plt.ylabel('Predicciones')
plt.title(f'Ecuación Normal: Predicciones vs Reales\nMSE={mse_normal:.4f}, R²={r2_normal:.4f}')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_sk, alpha=0.5, edgecolors='k', linewidth=0.5, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Valores Reales')
plt.ylabel('Predicciones')
plt.title(f'sklearn: Predicciones vs Reales\nMSE={mse_sk:.4f}, R²={r2_sk:.4f}')
plt.grid(True)

plt.suptitle('Comparación de Modelos: Predicciones vs Valores Reales', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Análisis de residuos

Los residuos ($e_i = y_i - \hat{y}_i$) deben tener media cero y no mostrar patrones claros si el modelo es adecuado.

In [ ]:
residuos_normal = y_test - y_pred_normal
residuos_sk = y_test - y_pred_sk

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_pred_normal, residuos_normal, alpha=0.5, edgecolors='k', linewidth=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Residuos')
plt.title('Residuos vs Predicciones - Ecuación Normal')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(y_pred_sk, residuos_sk, alpha=0.5, edgecolors='k', linewidth=0.5, color='green')
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Residuos')
plt.title('Residuos vs Predicciones - sklearn')
plt.grid(True)

plt.suptitle('Análisis de Residuos', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Media de residuos (normal): {np.mean(residuos_normal):.4e}")
print(f"Media de residuos (sklearn): {np.mean(residuos_sk):.4e}")
print("✅ La media de residuos cercana a cero indica que el modelo es insesgado.")

### Interpretación de $R^2$

El $R^2$ mide la proporción de la varianza de la variable dependiente que es predecible a partir de las variables independientes.

*   $R^2 = 1$: Predicción perfecta.
*   $R^2 = 0$: El modelo no explica nada (equivale a predecir siempre la media).
*   $R^2 < 0$: El modelo es peor que predecir la media (puede ocurrir si no hay intercepto o si el modelo está muy mal especificado).

Nuestro $R^2 \approx 0.6$ en test indica que las características explican aproximadamente el 60% de la variabilidad en los precios de las viviendas.

---
## Ejercicios Adicionales para el Estudiante

1.  **Sin intercepto:** Vuelve a calcular la ecuación normal **sin** incluir la columna de unos (es decir, forzando a que la recta pase por el origen). ¿Cómo cambian las métricas? ¿Tiene sentido en este contexto?

2.  **Regularización (opcional):** Investiga la **regresión Ridge** (mínimos cuadrados con regularización L2). La solución es $\hat{\beta} = (X^T X + \lambda I)^{-1} X^T y$. Implementa Ridge con $\lambda = 0.1$ y compara los coeficientes.

3.  **Características más importantes:** ¿Qué característica parece tener el coeficiente más grande (en valor absoluto)? ¿Tiene sentido intuitivo? (Por ejemplo, 'MedInc' - ingreso medio - debería estar positivamente correlacionado con el precio).

4.  **Validación cruzada:** Usa `cross_val_score` de sklearn para evaluar el modelo con validación cruzada de 5 folds. ¿Cómo se comparan los resultados con los obtenidos en train/test?

5.  **Reflexión:** La ecuación normal requiere invertir $X^T X$, lo que tiene costo $O(n^3)$ donde $n$ es el número de características. ¿Qué problemas podría tener esto si tuviéramos miles de características? ¿Qué alternativas existen?

---
## Conclusiones de la Sesión

*   Hemos implementado la **ecuación normal** desde cero, conectando directamente la teoría de sistemas sobredeterminados con la práctica de machine learning.
*   Nuestra implementación produce coeficientes **prácticamente idénticos** a los de `sklearn.linear_model.LinearRegression`, validando así nuestro código.
*   Evaluamos el modelo usando **MSE** y **$R^2$**, dos métricas fundamentales en regresión.
*   Visualizamos las predicciones vs valores reales y analizamos los **residuos**, verificando que el modelo es insesgado.
*   El dataset California Housing nos permitió trabajar con un problema real de predicción de precios, con múltiples características y cierta complejidad.
*   Este ejercicio consolida la conexión entre álgebra lineal (sistemas sobredeterminados, ecuación normal) y el primer modelo de machine learning: la regresión lineal.

¡Has completado tu primera implementación completa de un modelo de regresión desde cero!